# FABLE architecture pipeline: a fake-replay walkthrough

This notebook sends one small complex event through FABLE's real architectural components. The deployment, message broker, container runtime, and replay provider are deterministic in-process substitutes, so no camera, MQTT server, Docker daemon, or GPU is required.

The walkthrough exposes the main contracts instead of hiding them behind a convenience API: `SemanticGraph` → active frontier → `PredicateDemand` → alternatives and `ExecutionPlan` → admission and activation → `NodeAgent` result → updated semantic runtime.

In [ ]:
import sys
print(sys.executable)
print(sys.version)


In [2]:
from datetime import timedelta
from pathlib import Path
import sys
import tempfile
import time

# Jupyter normally starts this notebook in examples/, not at the repository
# root. Locate the checkout so the local fable package is importable without
# requiring every reader to register a repository-specific kernel.
search_roots = (Path.cwd(), *Path.cwd().parents)
REPO_ROOT = next(
    (path for path in search_roots if (path / "pyproject.toml").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from within a FABLE repository checkout")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pprint import pprint

from fable.common.time import EventTimeInterval, utc_now
from fable.distributed.config import ProviderRuntimeResolver
from fable.distributed.docker_runtime import FakeContainerRuntime
from fable.distributed.heartbeat import CapacitySampler, ReplaySourceProgressTracker
from fable.distributed.models import ProviderRuntimeSpec, RuntimeMode
from fable.distributed.node_agent import NodeAgent
from fable.distributed.orchestrator import DistributedOrchestrator
from fable.distributed.outbox import SQLiteOutbox, SQLiteProcessedLedger
from fable.distributed.persistence import InMemoryStateStore
from fable.distributed.transport import InMemoryBroker, InMemoryTransport, ReliableMessenger
from fable.integrations.reference_runtime import SyntheticReferenceRuntime
from fable.integrations.replay import build_replay_output_adapter_registry
from fable.planning import (
    ArtifactCatalog, BoundedLabelPlanner, DemandCompileContext, DemandCompiler,
    PhysicalAlternativeGraphBuilder, PhysicalPlanner, default_predicate_registry,
)
from fable.planning.testing import fake_deployment, fake_provider_registry
from fable.scheduling.adapters import candidate_from_search_result
from fable.scheduling.admission import MultiTenantScheduler
from fable.scheduling.capacity import CapacityLedger
from fable.scheduling.lifecycle import ProviderLifecycleManager
from fable.scheduling.models import TaskSchedulingPolicy
from fable.semantic import SemanticRuntime, SemanticRuntimeConfig
from fable.semantic.authoring import ComplexEvent

print("Using FABLE checkout:", REPO_ROOT)
DEMO_TIME = utc_now()

def show_model(value):
    """Render a Pydantic contract as readable JSON-compatible data."""
    pprint(value.model_dump(mode="json", exclude_none=True), sort_dicts=False)

ImportError: cannot import name 'StrEnum' from 'enum' (/usr/lib/python3.10/enum.py)

## 1. Author a provider-independent event graph

The new authoring API describes *what evidence means*, not where a model runs. This event completes when a microphone reports an audio event labeled `door_slam` in the loading zone.

In [ ]:
# ComplexEvent is the provider-independent authoring surface. Nothing in
# this block selects a microphone, classifier, machine, or deployment.
event = ComplexEvent(
    "door_slam_demo",
    namespace="fable.examples.notebook",
    name="Door slam at loading zone",
)
# Event-level roles are variables that predicates can bind or validate.
event.role("place", "zone")
door_slam = event.predicate(
    "AUDIO_EVENT",
    bind={"location": "place"},
    parameters={"label": "door_slam", "minimum_confidence": 0.7},
    key="door_slam",
    checkpoint=True,
)
# Selecting door_slam as the root compiles and validates an immutable DAG.
graph = door_slam.build()
show_model(graph)

## 2. Start the semantic runtime and inspect its active frontier

Starting the runtime creates an initial hypothesis. The frontier is the currently actionable semantic checkpoint: the evidence FABLE needs next.

In [ ]:
# One SemanticRuntime owns the hypotheses and frontiers for this request.
# It evaluates evidence, but it does not choose or invoke providers.
runtime = SemanticRuntime(
    graph,
    config=SemanticRuntimeConfig(request_id="notebook-door-slam"),
)
# start() creates the initial hypothesis over the requested event-time window.
transition = runtime.start(
    event_time_window=EventTimeInterval(
        start=DEMO_TIME - timedelta(seconds=5),
        end=DEMO_TIME + timedelta(seconds=30),
    ),
    observed_at=DEMO_TIME,
)
# A transition names every hypothesis/frontier changed by the operation.
hypothesis = runtime.get_hypothesis(transition.hypothesis_ids[0])
frontier = transition.frontiers[0]
print("Hypothesis lifecycle:", hypothesis.lifecycle.value)
show_model(frontier)

## 3. Compile the frontier into a physical demand

The demand compiler combines the semantic checkpoint with deployment constraints. Here we explicitly select the fake microphone on `sensor_a` and require raw audio to remain local.

In [ ]:
# These fixtures describe available machines, sensors, links, provider
# contracts, and data types. They contain no running provider processes.
deployment = fake_deployment()
providers = fake_provider_registry()
artifacts = ArtifactCatalog()  # No reusable intermediate artifacts yet.
compiler = DemandCompiler(
    predicate_registry=default_predicate_registry(),
    deployment=deployment,
)
# Compilation translates semantic intent into a physical requirement while
# retaining the hypothesis/frontier/checkpoint IDs needed on the result path.
demands = compiler.compile(
    graph=runtime.graph,
    hypothesis=hypothesis,
    frontier=frontier,
    context=DemandCompileContext(
        # Map this graph node to the only source that can observe it.
        eligible_source_ids_by_node={
            runtime.graph.nodes_by_key["door_slam"].node_id: ("microphone_store",),
        },
        # Force classification beside the microphone rather than transferring audio.
        raw_data_must_remain_local=True,
        allowed_node_ids=("sensor_a",),
    ),
)
demand = demands[0]
show_model(demand)

## 4. Generate alternatives and choose an execution plan

Alternative generation enumerates valid provider/data-placement chains. The bounded planner scores those alternatives and returns a self-contained execution plan for the scheduler.

In [ ]:
# Phase A: enumerate legal provider chains, placements, inputs, and transfers.
alternative_builder = PhysicalAlternativeGraphBuilder(
    provider_registry=providers, artifact_catalog=artifacts, deployment=deployment
)
# Phase B: keep a bounded Pareto set and choose among feasible alternatives.
bounded_planner = BoundedLabelPlanner(
    provider_registry=providers, artifact_catalog=artifacts, deployment=deployment
)
# PhysicalPlanner composes enumeration and search into one planning call.
planned = PhysicalPlanner(
    alternative_generator=alternative_builder, plan_search=bounded_planner
).plan(demands, now=DEMO_TIME)
print("Feasible alternatives:", len(planned.alternatives.alternatives))
show_model(planned.execution_plan)

## 5. Convert the plan into a schedulable candidate

The adapter removes planner-internal details and produces the contract consumed by admission and provider lifecycle management.

In [ ]:
# The scheduler does not consume the planner's search labels directly. This
# adapter packages the selected ExecutionPlan, demands, costs, and fallback rank.
candidate = candidate_from_search_result(
    planned.search,
    planned.alternatives,
    demands,
    task_policy=TaskSchedulingPolicy(request_id=hypothesis.request_id),
)
show_model(candidate)

## 6. Admit and execute through a real node agent

This cell assembles the distributed substrate explicitly. `InMemoryBroker` replaces MQTT, `FakeContainerRuntime` replaces Docker, and `SyntheticReferenceRuntime` emits deterministic fake replay evidence. Scheduling, leases, reliable messages, activation commands, and `NodeAgent` handling are the normal FABLE implementations.

In [ ]:
# Durable outboxes and duplicate-delivery ledgers normally survive process
# restarts. The notebook keeps their SQLite files in a temporary directory.
scratch = tempfile.TemporaryDirectory(prefix="fable-notebook-")
state_root = Path(scratch.name)
# Every transport below shares this broker. It replaces an external MQTT
# service but preserves topic subscriptions and serialized message delivery.
broker = InMemoryBroker()
store = InMemoryStateStore()  # Captures tasks, plans, results, and artifacts.
received_results = []  # Callback sink used in the next notebook section.

# The lifecycle manager owns capacity reservations, provider instances, and
# leases. The scheduler decides whether this candidate can be admitted now.
lifecycle = ProviderLifecycleManager(
    provider_registry=providers, capacity=CapacityLedger(deployment)
)
scheduler = MultiTenantScheduler(lifecycle=lifecycle)

# The orchestrator is the control-plane side of distributed execution. It
# asks the scheduler to admit plans and translates admitted steps into
# provider activation commands addressed to individual node agents.
orchestrator_transport = InMemoryTransport(broker)
orchestrator = DistributedOrchestrator(
    orchestrator_id="notebook-orchestrator",
    transport=orchestrator_transport,
    messenger=ReliableMessenger(
        entity_id="notebook-orchestrator", transport=orchestrator_transport,
        outbox=SQLiteOutbox(state_root / "orchestrator-outbox.sqlite"), retry_interval=100,
    ),
    processed_ledger=SQLiteProcessedLedger(state_root / "orchestrator-processed.sqlite"),
    store=store, scheduler=scheduler, lifecycle=lifecycle,
    # A runtime resolver maps the logical provider selected by planning to
    # its node-local launch mode. REFERENCE means deterministic fake replay.
    runtime_resolver=ProviderRuntimeResolver({
        ("sensor_a", "audio_event_classifier"): ProviderRuntimeSpec(
            provider_id="audio_event_classifier", provider_contract_version=1,
            node_id="sensor_a", mode=RuntimeMode.REFERENCE, reference_delay_ms=25,
        )
    }),
    # In the deployed controller this callback feeds SemanticRuntime.apply().
    # We retain the result briefly so the notebook can display it first.
    on_result=received_results.append, monitor_interval=100,
)
# This is a real FABLE NodeAgent for the planner-selected sensor_a node. It
# receives activation commands, manages provider state, and publishes typed
# results. Only its external container/provider integrations are fakes.
agent_transport = InMemoryTransport(broker)
agent = NodeAgent(
    node_id="sensor_a", session_id="notebook-session", transport=agent_transport,
    messenger=ReliableMessenger(
        entity_id="sensor_a", transport=agent_transport,
        outbox=SQLiteOutbox(state_root / "agent-outbox.sqlite"), retry_interval=100,
    ),
    processed_ledger=SQLiteProcessedLedger(state_root / "agent-processed.sqlite"),
    # Avoid Docker while retaining the same node-agent lifecycle interface.
    container_runtime=FakeContainerRuntime(),
    progress=ReplaySourceProgressTracker(node_id="sensor_a"),
    state_dir=state_root / "sensor_a", heartbeat_interval=100,
    capacity_sampler=CapacitySampler(gpu_free_mb_override=8192),
    # Adapters normalize provider output; the reference runtime generates a
    # deterministic PredicateResult instead of running an audio classifier.
    output_adapters=build_replay_output_adapter_registry(),
    reference_runtime=SyntheticReferenceRuntime(),
)
# start() subscribes both components and starts their reliability/heartbeat
# workers. Starting them before submission prevents losing the first command.
orchestrator.start()
agent.start()

# submit_candidates() crosses the scheduler boundary. On admission it creates
# reservations and leases, persists the plan, and publishes one activation
# command per executable step. The node agent consumes that command
# asynchronously through the shared broker.
batch, activation_commands = orchestrator.submit_candidates((candidate,), now=DEMO_TIME)
print("Admission status:", batch.records[0].decision.value)
print("Selected node agent:", activation_commands[0].node_id)
show_model(activation_commands[0])

## 7. Receive fake replay evidence and update semantics

The reference provider runs behind the node agent and returns a normal typed `PredicateResult`. Applying it to the semantic runtime resolves the active frontier and completes this one-predicate event.

In [ ]:
# Provider execution is asynchronous, so poll with a bounded deadline rather
# than assuming the result is available immediately after plan submission.
deadline = time.monotonic() + 2.0
while not received_results and time.monotonic() < deadline:
    time.sleep(0.01)
assert received_results, "the fake replay provider did not return a result"
# This is the same transportable PredicateResult contract a real provider
# would emit; it carries request, hypothesis, checkpoint, and provenance IDs.
result = received_results[0]
show_model(result)

# Applying evidence closes the loop. SemanticRuntime validates that the result
# targets the active checkpoint, updates truth/bindings, and derives frontiers.
semantic_update = runtime.apply(result)
updated = runtime.get_hypothesis(semantic_update.hypothesis_ids[0])
print("Transition status:", semantic_update.status.value)
print("Updated hypothesis lifecycle:", updated.lifecycle.value)
print("Remaining active frontiers:", len(semantic_update.frontiers))
show_model(updated)

## 8. Clean up

The fake transports use background threads and SQLite outboxes, so stop them when experimenting interactively.

In [ ]:
# Stop subscribers and worker threads before deleting their durable state.
agent.stop()
orchestrator.stop()
scratch.cleanup()
print("Notebook pipeline complete.")